In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data_2020 = pd.read_csv('../data/senate_general_indiv20.csv', dtype={'ZIP_CODE':str})
data_2022 = pd.read_csv('../data/senate_general_indiv22.csv', dtype={'ZIP_CODE':str})

data_2020['YEAR'] = '2020'
data_2022['YEAR'] = '2022'

print('2020 shape: ', data_2020.shape)
print('2022 shape: ', data_2022.shape)

agg = pd.concat([data_2020, data_2022])
print('combined shape: ', agg.shape)

agg['ZIP_CODE'] = agg['ZIP_CODE'].astype(str).str.zfill(5)
agg.head()

2020 shape:  (2232708, 16)
2022 shape:  (2017528, 16)
combined shape:  (4250236, 16)


,CMTE_ID,IMAGE_NUM,TRANSACTION_TP,NAME,CITY,STATE,ZIP_CODE,EMPLOYER,OCCUPATION,TRANSACTION_DT,TRANSACTION_AMT,CAND_ID,CAND_NAME,CAND_PTY_AFFILIATION,CAND_OFFICE_ST,YEAR
0,C00647842,202010089285059826,15,"LOMBARDO, ROBERT",MANCHESTER,NH,03102,ELECTRONICS FOR IMAGING INC,TRADE COMPLIANCE MANAGER,2020-09-08,275,S0NH00300,"O'DONNELL, JUSTIN F",LIB,NH,2020
1,C00716340,202010089285059649,15,"FLOYD, WILLIAM",EVANSTON,IL,60201,RETIRED,RETIRED,2020-09-10,500,S0IL00543,"CURRAN, MARK",REP,IL,2020
2,C00716340,202010089285059651,15,"GULLY, MICHAEL",QUINCY,IL,62305,GULLY TRANSPORTATION,TRUCK LINE EXECUTIVE,2020-09-07,1000,S0IL00543,"CURRAN, MARK",REP,IL,2020
3,C00716340,202010089285059651,15,"GULLY, MICHAEL",QUINCY,IL,62305,GULLY TRANSPORTATION,TRUCK LINE EXECUTIVE,2020-09-07,250,S0IL00543,"CURRAN, MARK",REP,IL,2020
4,C00716340,202010089285059661,15,"O'BRIEN, KATHLEEN",LAKE FOREST,IL,60045,SELF,ATTORNEY,2020-09-02,250,S0IL00543,"CURRAN, MARK",REP,IL,2020


In [3]:
# fix zip codes
pre_rows = len(agg)
agg = agg.dropna(subset=['ZIP_CODE'])
print('Rows removed from missingness: ', pre_rows - len(agg))

pre_rows = len(agg)
agg = agg[~agg['ZIP_CODE'].str.contains(r'[a-zA-Z]', na=False)]
print('Rows removed from letters: ', pre_rows - len(agg))

agg['ZIP_CODE'] = agg['ZIP_CODE'].where(agg['ZIP_CODE'].str.len() != 9, agg['ZIP_CODE'].str[:5])

pre_rows = len(agg)
agg = agg[agg['ZIP_CODE'].str.len() == 5]
print('Rows removed from not five digits: ', pre_rows - len(agg))

print('Remaining rows: ', len(agg))

Rows removed from missingness:  527
Rows removed from letters:  33
Rows removed from not five digits:  4
Remaining rows:  4249672


In [4]:
census = pd.read_csv('../data/census_cleaned.csv', dtype={'zip_code':str})
census["zip_code"] = census["zip_code"].astype(str).str.zfill(5)

census.head()

,zip_code,population,median_income
0,00601,17242,19454
1,00602,37548,21420
2,00603,49804,20933
3,00606,5009,20992
4,00610,25731,24496


In [5]:
education = pd.read_csv('../data/zip_to_education.csv', dtype={'zip':str})

education.head()

,zip,educ_attainment_pop_18plus,college_degree_or_higher_count_18plus,prop_college_degree_or_higher_18plus,high_school_or_higher_count_18plus,prop_high_school_or_higher_18plus
0,00601,13764,2346,0.170445,10004,0.726824
1,00602,31673,7140,0.225429,23100,0.729328
2,00603,40674,10698,0.263018,32476,0.798446
3,00606,4338,526,0.121254,2637,0.607884
4,00610,21548,4969,0.230601,16585,0.769677


In [6]:
election_diff = pd.read_csv('../data/zip_to_election_diff.csv', dtype={'ZIP_CODE':str})

election_diff.head()

,ZIP_CODE,election_2020,election_2022,election_diff
0,00602,0,0,0
1,00603,0,0,0
2,00612,0,0,0
3,00616,0,0,0
4,00624,0,0,0


In [8]:
# represents the total donations originating from zip to each (party, state) by year
print('===== CLEANING ZIP CODES =====')
print('(and checking how much data is lost)')

zip_df = agg.groupby(['YEAR', 'ZIP_CODE', 'STATE', 'CAND_OFFICE_ST', 'CAND_PTY_AFFILIATION']).agg({'TRANSACTION_AMT':'sum'})
zip_df.reset_index(inplace=True)

print('\nnum zip codes to start (possibly including errors): ', zip_df['ZIP_CODE'].nunique())
print('num data points: ', len(zip_df))
print('total donation amount: ', zip_df['TRANSACTION_AMT'].sum())

zip_df = pd.merge(zip_df,
         census,
         left_on='ZIP_CODE',
         right_on='zip_code',
         how='inner')
zip_df.drop(columns=['zip_code'], inplace=True)

print('\nnum zip codes after joining population and income data: ', zip_df['ZIP_CODE'].nunique())
print('num data points: ', len(zip_df))
print('total donation amount: ', zip_df['TRANSACTION_AMT'].sum())

zip_df = pd.merge(zip_df,
         education,
         left_on='ZIP_CODE',
         right_on='zip',
         how='inner')
zip_df.drop(columns=['zip'], inplace=True)

print('\nnum zip codes after joining education data: ', zip_df['ZIP_CODE'].nunique())
print('num data points: ', len(zip_df))
print('total donation amount: ', zip_df['TRANSACTION_AMT'].sum())

zip_df = pd.merge(zip_df,
         election_diff,
         left_on='ZIP_CODE',
         right_on='ZIP_CODE',
         how='inner')

print('\nnum zip codes after joining election status data: ', zip_df['ZIP_CODE'].nunique())
print('num data points: ', len(zip_df))
print('total donation amount: ', zip_df['TRANSACTION_AMT'].sum())

print('\n--- checking zip code alignment with state ---')

import pgeocode

nomi = pgeocode.Nominatim("us")
zip_lookup = nomi.query_postal_code(zip_df["ZIP_CODE"].astype(str).str.zfill(5).tolist())

zip_df["ZIP_STATE_MATCH"] = (
    zip_lookup["state_code"].to_numpy() == zip_df["STATE"].str.upper().to_numpy()
)

mismatches = zip_df[~zip_df["ZIP_STATE_MATCH"]]

print('num mismatched zip codes: ', mismatches['ZIP_CODE'].nunique())
print('total donations in mismatched rows: ', mismatches['TRANSACTION_AMT'].sum())

n_before = len(zip_df)

zip_df = zip_df[zip_df["ZIP_STATE_MATCH"]].copy()
zip_df = zip_df.drop(columns=["ZIP_STATE_MATCH"])

n_dropped = n_before - len(zip_df)
print(f"Dropped {n_dropped:,} mismatched rows")
print('----------------------------------------------')

print('\nfinal num zip codes: ', zip_df['ZIP_CODE'].nunique())
print('num data points: ', len(zip_df))
print('total donation amount: ', zip_df['TRANSACTION_AMT'].sum())

zip_df.head(20)

===== CLEANING ZIP CODES =====
(and checking how much data is lost)

num zip codes to start (possibly including errors):  28103
num data points:  345570
total donation amount:  648532228

num zip codes after joining population and income data:  23549
num data points:  322465
total donation amount:  607619019

num zip codes after joining education data:  23549
num data points:  322465
total donation amount:  607619019

num zip codes after joining election status data:  18041
num data points:  307771
total donation amount:  601598388

--- checking zip code alignment with state ---
num mismatched zip codes:  477
total donations in mismatched rows:  442452
Dropped 648 mismatched rows
----------------------------------------------

final num zip codes:  17985
num data points:  307123
total donation amount:  601155936


,YEAR,ZIP_CODE,STATE,CAND_OFFICE_ST,CAND_PTY_AFFILIATION,TRANSACTION_AMT,population,median_income,educ_attainment_pop_18plus,college_degree_or_higher_count_18plus,prop_college_degree_or_higher_18plus,high_school_or_higher_count_18plus,prop_high_school_or_higher_18plus,election_2020,election_2022,election_diff
76,2020,01001,MA,AZ,DEM,330,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1
77,2020,01001,MA,AZ,REP,290,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1
78,2020,01001,MA,CO,DEM,250,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1
79,2020,01001,MA,GA,DEM,260,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1
80,2020,01001,MA,IA,DEM,350,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1
81,2020,01001,MA,KS,DEM,125,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1
82,2020,01001,MA,KY,DEM,20,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1
83,2020,01001,MA,KY,REP,45,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1
84,2020,01001,MA,MA,DEM,100,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1
85,2020,01001,MA,ME,DEM,300,16984,75342,13438,4639,0.345215,12605,0.938012,1,0,-1


In [9]:
# add geodata
import pgeocode

nomi = pgeocode.Nominatim('us')
geo_info = nomi.query_postal_code(zip_df['ZIP_CODE'].tolist())

zip_df['lat'] = geo_info['latitude'].values
zip_df['lon'] = geo_info['longitude'].values

zip_df.head()

# drop if no geodata
pre_len = len(zip_df)
zip_df = zip_df.dropna(subset=['lat', 'lon'])
print('Number removed for missing geodata: ', pre_len - len(zip_df))

# drop if not in contiguous USA
pre_len = len(zip_df)
zip_df = zip_df[
    (zip_df['lat'] >= 24.5) & (zip_df['lat'] <= 49.5) &
    (zip_df['lon'] >= -125.0) & (zip_df['lon'] <= -66.0)
]
print('Number removed for outside contiguous USA: ', pre_len - len(zip_df))

Number removed for missing geodata:  0
Number removed for outside contiguous USA:  2605


In [ ]:
zip_df.to_csv('senate_campaign_data_by_zip.csv', index=False)